# Root-cause analysis: which inputs drive the price after a yield shift?

This notebook trains the pricing network on simulated baseline data, then measures how sensitive its prices are to each input on a sample where the 10-year yield has moved from about 4.1% to 4.9%. All data is simulated.

Outputs are not saved in the repository. Run the cells to produce them.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import torch

from src.data.simulate import FEATURE_COLUMNS, generate_pricing_data
from src.models.train import evaluate, train_model

## 1. Train on baseline data

In [ ]:
baseline = generate_pricing_data(n_rows=3000, seed=42)
model, scaler = train_model(baseline, epochs=200, seed=0)
print(f"Baseline RMSE: {evaluate(model, scaler, baseline)['rmse']:.3f}")

## 2. Gradients of the price on a drifted sample

The gradient of the predicted price with respect to each input shows how much that input moves the price. Inputs are in the model's scaled units.

In [ ]:
drifted = generate_pricing_data(n_rows=200, seed=7, yield_mean=0.049)
print(f"RMSE on the drifted sample: {evaluate(model, scaler, drifted)['rmse']:.3f}")

x, macro, _ = scaler.transform(drifted)
features = torch.from_numpy(x).requires_grad_(True)
yields = torch.from_numpy(macro).requires_grad_(True)
model(features, yields).sum().backward()

feature_grad = np.asarray(features.grad)
yield_grad = np.asarray(yields.grad)

## 3. Summary

In [ ]:
for name, value in zip(FEATURE_COLUMNS, np.abs(feature_grad).mean(axis=0)):
    print(f"{name}: mean absolute sensitivity {value:.3f}")
print(f"Yield input: mean sensitivity {yield_grad.mean():.3f}")
print(f"Rows where a higher yield lowers the price: {(yield_grad < 0).mean():.0%}")

A negative yield sensitivity on most rows is what the simulation is built to produce: prices fall when yields rise. If the drifted-sample RMSE is much higher than the baseline RMSE, run `python pipelines/intraday_recalibration.py`, which refits only the macro adjustment layer.